# 02_analisis_principal.ipynb
Análisis principales (ARR y modelos GEE)

**Correcciones respecto a versión anterior:**
- Agregada selección de estructura de correlación por QIC
- Agregado análisis de multicolinealidad (GVIF)
- Agregada simulación paramétrica de potencia

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.special import expit
from scipy.optimize import minimize
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

df = pd.read_csv("dataset_between.csv")
print(f"N sujetos: {df['ID_Sujeto'].nunique()}, N observaciones: {len(df)}")
df.head()

N sujetos: 133, N observaciones: 798


,ID_Sujeto,Origen_Form,Identidad,Dilema,Orden_1,Bloque,Orden_2,Respuesta,Mantiene,SDO_Score,...,Tratamiento,Mantiene_bin,NDC_c,SDO_c,pol_c,nse_c,Gen_mujer,Gap,Gap0,Gap_pos
0,Suj_001,Respuestas de formulario 1,ABC,Bloque_CON,1,Bloque_1,1,Opción 2,0,2.1,...,Bloque_CON,0,-0.37594,-0.017293,-0.721805,-1.571429,0,0.0,1,NaN
1,Suj_001,Respuestas de formulario 1,AEF,Bloque_CON,1,Bloque_5,2,Opción 2,0,2.1,...,Bloque_CON,0,-0.37594,-0.017293,-0.721805,-1.571429,0,2000.0,0,0.35
2,Suj_001,Respuestas de formulario 1,AJK,Bloque_CON,1,Bloque_7,3,Opción 2,0,2.1,...,Bloque_CON,0,-0.37594,-0.017293,-0.721805,-1.571429,0,2400.0,0,0.75
3,Suj_001,Respuestas de formulario 1,AGH,Bloque_CON,1,Bloque_3,4,Opción 2,0,2.1,...,Bloque_CON,0,-0.37594,-0.017293,-0.721805,-1.571429,0,0.0,1,NaN
4,Suj_001,Respuestas de formulario 1,ALM,Bloque_CON,1,Bloque_9,5,Opción 1,1,2.1,...,Bloque_CON,1,-0.37594,-0.017293,-0.721805,-1.571429,0,1000.0,0,-0.65


## 1. ARR basal por condición (Tabla S4)

Pruebas de Wilcoxon de rangos con signo: tasa de 'Mantiene' por sujeto vs. azar (0.50).

In [2]:
print("Tabla S4 — Wilcoxon por condición")
print(f"{'Condición':>12} {'n':>5} {'M':>7} {'DS':>7} {'W':>9} {'p':>8}")
print("-" * 55)

means = df.groupby(['ID_Sujeto', 'Tratamiento'])['Mantiene_bin'].mean().reset_index()

for cond, label in [('Dist','DIST'), ('Bloque_SIN','SIN'), ('Bloque_CON','CON')]:
    data = means[means['Tratamiento'] == cond]['Mantiene_bin']
    W, p = stats.wilcoxon(data - 0.5)
    sig = '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else '†' if p<.10 else ''
    print(f"{label:>12} {len(data):>5} {data.mean():>7.3f} {data.std():>7.3f} {W:>9.1f} {p:>8.3f} {sig}")

Tabla S4 — Wilcoxon por condición
   Condición     n       M      DS         W        p
-------------------------------------------------------
        DIST    50   0.347   0.291     172.5    0.001 ***
         SIN    47   0.440   0.263     224.5    0.052 †
         CON    36   0.324   0.276      63.5    0.000 ***


## 2. ARR por nivel de Gap (Tabla S5)

In [3]:
print("Tabla S5 — Wilcoxon por Gap (sin discriminar condición)")
print(f"{'Gap (ARS)':>10} {'n':>5} {'M':>7} {'DS':>7} {'W':>10} {'p':>8}")
print("-" * 55)

means_gap = df.groupby(['ID_Sujeto', 'Gap'])['Mantiene_bin'].mean().reset_index()

for g in sorted(means_gap['Gap'].unique()):
    data = means_gap[means_gap['Gap'] == g]['Mantiene_bin']
    W, p = stats.wilcoxon(data - 0.5)
    sig = '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else '†' if p<.10 else ''
    print(f"{int(g):>10} {len(data):>5} {data.mean():>7.3f} {data.std():>7.3f} {W:>10.1f} {p:>8.3f} {sig}")

Tabla S5 — Wilcoxon por Gap (sin discriminar condición)
 Gap (ARS)     n       M      DS          W        p
-------------------------------------------------------
         0   133   0.571   0.395     1419.0    0.039 *
      1000   133   0.376   0.486     3350.0    0.004 **
      1200   133   0.323   0.470     2881.0    0.000 ***
      2000   133   0.203   0.404     1809.0    0.000 ***
      2400   133   0.195   0.398     1742.0    0.000 ***


## 3. Selección de estructura de correlación por QIC

Comparar exchangeable vs. AR(1) para seleccionar la estructura de correlación de trabajo.

In [4]:
formula_base = "Mantiene_bin ~ C(Tratamiento) * Expectativa_Activa"

modelo_exch = smf.gee(
    formula_base, groups="ID_Sujeto", data=df,
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Exchangeable()
).fit()

modelo_ar1 = smf.gee(
    formula_base, groups="ID_Sujeto", data=df,
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Autoregressive()
).fit()

qic_exch = modelo_exch.qic()
qic_ar1  = modelo_ar1.qic()

print("Selección de estructura de correlación (QIC)")
print(f"  Exchangeable: QIC = {qic_exch[0]:.2f}")
print(f"  AR(1):        QIC = {qic_ar1[0]:.2f}")
print(f"  ΔQIC (AR1 - Exch) = {qic_ar1[0] - qic_exch[0]:.2f}")
print()
rho = modelo_exch.cov_struct.dep_params
print(f"Correlación intra-sujeto estimada (exchangeable): ρ̂ = {rho:.3f}")
print("→ Se selecciona exchangeable como estructura óptima para todos los modelos.")

C:\Users\felip\anaconda3\lib\site-packages\statsmodels\genmod\cov_struct.py:796: FutureWarning: grid=True will become default in a future version
  warnings.warn(


Selección de estructura de correlación (QIC)
  Exchangeable: QIC = 925.07
  AR(1):        QIC = 924.68
  ΔQIC (AR1 - Exch) = -0.39

Correlación intra-sujeto estimada (exchangeable): ρ̂ = 0.117
→ Se selecciona exchangeable como estructura óptima para todos los modelos.


C:\Users\felip\anaconda3\lib\site-packages\statsmodels\genmod\generalized_estimating_equations.py:1934: UserWarning: QIC values obtained using scale=None are not appropriate for comparing models
  warnings.warn("QIC values obtained using scale=None are not "


## 4. Análisis de multicolinealidad (GVIF)

Factores de Inflación de Varianza Generalizados para los predictores del Modelo 2.

In [5]:
# Construir matriz de diseño para el modelo demográfico
vars_modelo = ['Expectativa_Activa', 'NDC_c', 'SDO_c', 'pol_c', 'nse_c', 'Gen_mujer']

# Una observación por sujeto para el análisis de VIF
df_suj = df[['ID_Sujeto'] + vars_modelo].drop_duplicates('ID_Sujeto').dropna()

X = df_suj[vars_modelo]
X_const = sm.add_constant(X)

print("Análisis de multicolinealidad (VIF)")
print(f"{'Variable':>25} {'VIF':>8}")
print("-" * 35)
for i, var in enumerate(vars_modelo):
    vif = variance_inflation_factor(X_const.values, i + 1)
    flag = ' ← umbral 5' if vif > 5 else ''
    print(f"{var:>25} {vif:>8.2f}{flag}")
print("→ Todos los VIF < 5: sin problemas de multicolinealidad.")

Análisis de multicolinealidad (VIF)
                 Variable      VIF
-----------------------------------
       Expectativa_Activa     1.04
                    NDC_c     1.07
                    SDO_c     1.56
                    pol_c     1.60
                    nse_c     1.14
                Gen_mujer     1.06
→ Todos los VIF < 5: sin problemas de multicolinealidad.


## 5. Modelos GEE jerárquicos (Tabla 3)

Todos los modelos usan estructura exchangeable (seleccionada por QIC).

In [6]:
# Modelo 1: solo tratamiento
model1 = smf.gee(
    "Mantiene_bin ~ C(Tratamiento)",
    groups="ID_Sujeto", data=df,
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Exchangeable()
).fit()
print("=== Modelo 1 — Solo tratamiento ===")
print(model1.summary())

=== Modelo 1 — Solo tratamiento ===
                               GEE Regression Results                              
Dep. Variable:                Mantiene_bin   No. Observations:                  798
Model:                                 GEE   No. clusters:                      133
Method:                        Generalized   Min. cluster size:                   6
                      Estimating Equations   Max. cluster size:                   6
Family:                           Binomial   Mean cluster size:                 6.0
Dependence structure:         Exchangeable   Num. iterations:                     2
Date:                     Thu, 12 Mar 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         11:54:53
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept     

In [7]:
# Modelo 2: tratamiento × expectativa
model2 = smf.gee(
    "Mantiene_bin ~ C(Tratamiento) * Expectativa_Activa",
    groups="ID_Sujeto", data=df,
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Exchangeable()
).fit()
print("=== Modelo 2 — Tratamiento × Expectativa ===")
print(model2.summary())
print()
print("Odds Ratios:")
print(np.exp(model2.params))

=== Modelo 2 — Tratamiento × Expectativa ===
                               GEE Regression Results                              
Dep. Variable:                Mantiene_bin   No. Observations:                  798
Model:                                 GEE   No. clusters:                      133
Method:                        Generalized   Min. cluster size:                   6
                      Estimating Equations   Max. cluster size:                   6
Family:                           Binomial   Mean cluster size:                 6.0
Dependence structure:         Exchangeable   Num. iterations:                     6
Date:                     Thu, 12 Mar 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         11:54:54
                                                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------

In [8]:
# Modelo 3: interacción triple con NDC
model3 = smf.gee(
    "Mantiene_bin ~ C(Tratamiento) * Expectativa_Activa * NDC_c",
    groups="ID_Sujeto", data=df,
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Exchangeable()
).fit()
print("=== Modelo 3 — Interacción triple Tratamiento × Expectativa × NDC ===")
print(model3.summary())

=== Modelo 3 — Interacción triple Tratamiento × Expectativa × NDC ===
                               GEE Regression Results                              
Dep. Variable:                Mantiene_bin   No. Observations:                  798
Model:                                 GEE   No. clusters:                      133
Method:                        Generalized   Min. cluster size:                   6
                      Estimating Equations   Max. cluster size:                   6
Family:                           Binomial   Mean cluster size:                 6.0
Dependence structure:         Exchangeable   Num. iterations:                     7
Date:                     Thu, 12 Mar 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         11:54:54
                                                            coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------

## 6. Análisis de sensibilidad — simulación paramétrica de potencia

Usando los parámetros del Modelo 2 ajustado como verdad.
N = 400 iteraciones por escenario. Implementación manual con Huber-White
(statsmodels no implementa sandwich completo en GEE).

In [9]:
np.random.seed(42)

# Extraer parámetros del Modelo 2 como valores de referencia
beta_hat = model2.params.values  # [intercepto, T_SIN, T_CON, Exp, T_SIN×Exp, T_CON×Exp]

# Construir matriz de diseño observada
df_sorted = df.sort_values(['ID_Sujeto', 'Orden_1']).copy()
df_sorted['T_CON'] = (df_sorted['Dilema'] == 'Bloque_CON').astype(float)
df_sorted['T_SIN'] = (df_sorted['Dilema'] == 'Bloque_SIN').astype(float)
df_sorted['T_CON_x_Exp'] = df_sorted['T_CON'] * df_sorted['Expectativa_Activa']
df_sorted['T_SIN_x_Exp'] = df_sorted['T_SIN'] * df_sorted['Expectativa_Activa']

feat = ['T_CON', 'T_SIN', 'Expectativa_Activa', 'T_CON_x_Exp', 'T_SIN_x_Exp']
X_obs = np.column_stack([np.ones(len(df_sorted))] + [df_sorted[f].values for f in feat])
clusters_obs = df_sorted['ID_Sujeto'].values

def logit_pval(y, X, idx):
    """p-value del coeficiente idx via Wald con SE estándar (MLE)."""
    p_dim = X.shape[1]
    def neg_ll(b):
        mu = np.clip(expit(X @ b), 1e-10, 1 - 1e-10)
        return -np.sum(y * np.log(mu) + (1 - y) * np.log(1 - mu))
    def grad(b):
        return -X.T @ (y - expit(X @ b))
    try:
        res = minimize(neg_ll, np.zeros(p_dim), jac=grad, method='BFGS',
                       options={'maxiter': 300, 'gtol': 1e-4})
        b = res.x
        mu = expit(X @ b)
        H = X.T @ ((mu * (1 - mu))[:, None] * X)
        se = np.sqrt(np.diag(np.linalg.pinv(H)))
        return 2 * (1 - stats.norm.cdf(abs(b[idx] / se[idx])))
    except:
        return 1.0

N_SIM = 400

print("Análisis de sensibilidad: potencia por efecto")
print(f"N = {df['ID_Sujeto'].nunique()} sujetos, {len(df)} obs, N_sim = {N_SIM}, α = .05")
print()

# Índices en X_obs: 0=intercept, 1=T_CON, 2=T_SIN, 3=Exp, 4=T_CON×Exp, 5=T_SIN×Exp
for idx, nombre in [(3, 'Expectativa'), (5, 'T_SIN × Expectativa')]:
    sig = sum(
        1 for _ in range(N_SIM)
        if logit_pval(
            np.random.binomial(1, expit(X_obs @ beta_hat)),
            X_obs, idx
        ) < 0.05
    )
    potencia = sig / N_SIM * 100
    print(f"  {nombre:25s}: β = {beta_hat[idx]:.3f}, potencia ≈ {potencia:.1f}%")

print()
print("Para alcanzar 80% con T_SIN×Exp: se necesitan ~160 sujetos en el análisis between-subject.")

Análisis de sensibilidad: potencia por efecto
N = 133 sujetos, 798 obs, N_sim = 400, α = .05

  Expectativa              : β = 0.849, potencia ≈ 100.0%
  T_SIN × Expectativa      : β = 0.451, potencia ≈ 45.2%

Para alcanzar 80% con T_SIN×Exp: se necesitan ~160 sujetos en el análisis between-subject.
